# RAG Engine - C++ Build & Test

This notebook builds the full C++ RAG Engine on GPU.

**IMPORTANT:**
1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Restart runtime
3. Run all cells in order

**Expected build time:** ~20-30 minutes

In [ ]:
# Setup
import os
import subprocess

PROJECT_DIR = '/content/rag-engine'
VCPKG_DIR = '/content/vcpkg'

def run_cmd(cmd, timeout=600):
    result = subprocess.run(
        f'cd {PROJECT_DIR} && {cmd}',
        shell=True, capture_output=True, text=True, timeout=timeout
    )
    return result

os.makedirs(PROJECT_DIR, exist_ok=True)
print('Setup complete')

## 1. GPU Verification

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('GPU Ready!')

## 2. Install System Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq cmake build-essential git curl wget unzip libprotobuf-dev protobuf-compiler libuv1-dev g++ ninja-build 2>&1 | tail -5
print('System packages installed')

## 3. Clone Repository

In [ ]:
os.chdir('/content')
!rm -rf rag-engine
!git clone https://github.com/gumeeee/rag-engine.git 2>&1 | tail -3
!mkdir -p /content/rag-engine/build /content/rag-engine/data/corpus /content/rag-engine/models
print('Repository cloned')

## 4. Install vcpkg

In [ ]:
if not os.path.exists('/content/vcpkg'):
    !git clone https://github.com/Microsoft/vcpkg.git /content/vcpkg 2>&1 | tail -3
!cd /content/vcpkg && ./bootstrap-vcpkg.sh 2>&1 | tail -3
print('vcpkg ready')

## 5. Install C++ Dependencies

In [ ]:
!export VCPKG_ROOT=/content/vcpkg && /content/vcpkg/vcpkg install faiss-cpu onnxruntime libuv protobuf spdlog nlohmann-json --triplet x64-linux 2>&1 | tail -20
print('Dependencies installed')

## 6. Generate Sample Corpus

In [ ]:
docs = {
    'transformer.txt': 'The Transformer uses self-attention. Key: positional encoding, multi-head attention.',
    'bert.txt': 'BERT uses bidirectional transformers. Pre-training: masked LM and NSP.',
    'gpt.txt': 'GPT is autoregressive. Uses next-token prediction. GPT-3 has 175B params.',
    'rag.txt': 'RAG combines LLM with vector retrieval. Reduces hallucination.',
    'vector_search.txt': 'Vector search uses embeddings. FAISS enables billion-scale search.'
}
for name, content in docs.items():
    with open(f'{PROJECT_DIR}/data/corpus/{name}', 'w') as f:
        f.write(content)
print(f'Created {len(docs)} documents')

## 7. Generate Embeddings

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = list(docs.values())
embeddings = model.encode(texts, convert_to_numpy=True)

output = f'{PROJECT_DIR}/data/corpus.corpus'
with open(output, 'wb') as f:
    f.write(len(texts).to_bytes(4, 'little'))
    f.write(embeddings.shape[1].to_bytes(4, 'little'))
    embeddings.astype(np.float32).tofile(f)
print(f'Saved {len(texts)} embeddings')

## 8. Build C++ Project

In [ ]:
os.chdir(f'{PROJECT_DIR}/build')
result = subprocess.run(
    'cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_TOOLCHAIN_FILE=/content/vcpkg/scripts/buildsystems/vcpkg.cmake -DVCPKG_TARGET_TRIPLET=x64-linux',
    shell=True, capture_output=True, text=True, timeout=300
)
print(result.stdout[-2000:] if result.stdout else 'No output')
if result.returncode != 0:
    print('ERROR:', result.stderr[-1000:] if result.stderr else '')

In [ ]:
print('Building (may take 10+ minutes)...')
result = subprocess.run(
    'cmake --build . --config Release -j$(nproc)',
    shell=True, capture_output=True, text=True, timeout=1800
)
if result.returncode == 0:
    print('\n=== BUILD SUCCEEDED ===')
else:
    print('\n=== BUILD FAILED ===')
    print(result.stderr[-2000:] if result.stderr else '')

## 9. Run Tests (if build succeeded)

In [ ]:
import os
binary = f'{PROJECT_DIR}/build/rag-engine'
if os.path.exists(binary):
    print(f'Binary found: {binary}')
    print(f'Size: {os.path.getsize(binary)/1024/1024:.1f} MB')
    print('\nTo run: ./rag-engine --config ../config/default_config.pb.txt')
else:
    print('Build failed - see errors above')